In [30]:
@bind("person","csv useHeaders=true","./disk/assignment/output/csvs","person.csv").
@bind("personFeaturesI","csv useHeaders=true","./disk/assignment/data/high-resolution","person_features.csv").

@model("person", "['PersonId(ID):int','gender:string','smoker:boolean','height:double','waist:double','hip:double','waist_hip_ratio:double','sysBP:int','diaBP:int','weight:double','BMI:double','age:int']").
@model("personFeatures", "['MasterID(ID):int','Mesor_hat:double','AmplitudeHat:double','AcrophaseHat:double','Peak:double','Nadir:double','PeakTime:double','NadirTime:double','Phase:double','Morning:double','Afternoon:double','Evening:double','Night:double','MorningPeak:double','AfternoonPeak:double','EveningPeak:double','NightPeak:double','BimodalAfternoon:boolean','BimodalEvening:boolean']").
@model("phenotype","['phenotype_id(ID):string']").

phenoType("BMIOverweight").
phenoType("BMIObese").
phenoType("BMIUnderweight").
phenoType("BMINormal").
phenoType("Smoker"). %
phenoType("NonSmoker"). %
phenoType("LowBloodPressure").
phenoType("HyperState2BP").
phenoType("HyperStage1BP").
phenoType("PreHyperBP").
phenoType("NormalBP").
phenoType("LowBP").
phenoType("LowWaistToHipRatio").
phenoType("MediumWaistToHipRatio").
phenoType("HighWaistToHipRatio").

phenoTypeExport(PhenoTypeId):- phenoType(PhenoTypeId).

@model("riskFactorOf","['RiskFactorId:phenotype','PersonId:person']").
riskFactorOf("BMIUnderweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B < 18.5.
@model("healthyPhenotypeOf","['PhenoTypeId:phenotype','PersonId:person']").
healthyPhenotypeOf("BMIHealthy",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 18.5, B < 25.
riskFactorOf("BMIOverweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 25.0, B < 30 .
riskFactorOf("BMIObese",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 30.0.
riskFactorOf("Smoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #T.
healthyPhenotypeOf("NonSmoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #F.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 160.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 140, Sys < 160, Dia < 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 90, Sys < 160, Dia < 100.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 120, Sys < 140, Dia < 90.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 80, Sys < 140, Dia < 90.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 90, Sys < 120, Dia < 80.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 60, Sys < 120, Dia < 80.
riskFactorOf("LowBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys < 90, Dia < 60.
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.8, G = "Female".
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.95, G = "Male".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.8, WH <= 0.85, G = "Female".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.95, WH < 1.0, G = "Male".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.85, G = "Female".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH >= 1.0, G = "Male".

healthyPhenotypesOf(MasterId,Phenotypes) :- healthyPhenotypeOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype).
riskFactorsOf(MasterId,Phenotypes,Count) :- riskFactorOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype), Count=mcount(Phenotype).
phenotypesOf(MasterId,Healthy,RiskFactors,RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).
phenotypesOf(MasterId,Healthy,{},RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), not riskFactorsOf(MasterId,RiskFactors),RiskCount = 0.
phenotypesOf(MasterId,{},RiskFactors,RiskCount) :- not healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).

phenotypeCounts(Phenotype,Count) :- riskFactorOf(Phenotype,I),Count=mcount(I).
phenotypeCounts(Phenotype,Count) :- healthyPhenotypeOf(Phenotype,I),Count=mcount(I).
@output("phenotypeCounts").

@model("hormoneFeatureType", "['personFeatureType(ID):string']").
hormoneFeatureType("lowerTailMesor").
hormoneFeatureType("lowerTailPeak").
hormoneFeatureType("lowerTailPeakTime").
hormoneFeatureType("lowerTailNadirTime").
hormoneFeatureType("lowerTailMorning").
hormoneFeatureType("lowerTailAfternoon").
hormoneFeatureType("lowerTailEvening").
hormoneFeatureType("lowerTailNight").
hormoneFeatureType("lowerTailMorningPeak").
hormoneFeatureType("lowerTailAfternoonPeak").
hormoneFeatureType("lowerTailEveningPeak").
hormoneFeatureType("lowerTailNightPeak").
hormoneFeatureType("upperTailMesor").
hormoneFeatureType("upperTailPeak").
hormoneFeatureType("upperTailPeakTime").
hormoneFeatureType("upperTailNadirTime").
hormoneFeatureType("upperTailMorning").
hormoneFeatureType("upperTailAfternoon").
hormoneFeatureType("upperTailEvening").
hormoneFeatureType("upperTailNight").
hormoneFeatureType("upperTailMorningPeak").
hormoneFeatureType("upperTailAfternoonPeak").
hormoneFeatureType("upperTailEveningPeak").
hormoneFeatureType("upperTailNightPeak").
hormoneFeatureType("bimodalAfternoon").
hormoneFeatureType("bimodalEvening").

hormoneFeatureTypeExport(T) :- hormoneFeatureType(T).

@model("hasHormoneFeature", "['MasterID:int', 'featureType:hormoneFeatureType', 'hormone:string']").

hasHormoneFeature(MasterID, "lowerTailMesor", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    M < 1.19.

hasHormoneFeature(MasterID, "lowerTailPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    P < 3.97.

hasHormoneFeature(MasterID, "lowerTailPeakTime", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    PT < 5.67.

hasHormoneFeature(MasterID, "lowerTailNadirTime", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    NT < 0.34.

hasHormoneFeature(MasterID, "lowerTailNadirTime", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    NT > 22.00.

hasHormoneFeature(MasterID, "lowerTailMorning", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Morn < 2.33.

hasHormoneFeature(MasterID, "lowerTailAfternoon", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    After < 0.82.

hasHormoneFeature(MasterID, "lowerTailEvening", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Eve < 0.67.

hasHormoneFeature(MasterID, "lowerTailNight", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Ni < 0.3.

hasHormoneFeature(MasterID, "lowerTailMorningPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    MornPeak < 3.97.

hasHormoneFeature(MasterID, "lowerTailAfternoonPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    AfterPeak < 1.2.

hasHormoneFeature(MasterID, "lowerTailEveningPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    EvePeak < 1.03.

hasHormoneFeature(MasterID, "lowerTailNightPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    NiPeak < 0.91.

hasHormoneFeature(MasterID, "upperTailMesor", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening), M > 6.96.

hasHormoneFeature(MasterID, "upperTailPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    P > 21.74.

hasHormoneFeature(MasterID, "upperTailPeakTime", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    PT >  11.0.

hasHormoneFeature(MasterID, "upperTailNadirTime", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    NT > 22.79.

hasHormoneFeature(MasterID, "upperTailMorning", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Morn > 11.08.

hasHormoneFeature(MasterID, "upperTailAfternoon", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    After > 5.95.

hasHormoneFeature(MasterID, "upperTailEvening", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Eve > 4.55.

hasHormoneFeature(MasterID, "upperTailNight", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    Ni > 2.88.

hasHormoneFeature(MasterID, "upperTailMorningPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    MornPeak > 21.74.

hasHormoneFeature(MasterID, "upperTailAfternoonPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    AfterPeak > 10.11.

hasHormoneFeature(MasterID, "upperTailEveningPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    EvePeak > 6.74.

hasHormoneFeature(MasterID, "upperTailNightPeak", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    NiPeak > 10.26.

hasHormoneFeature(MasterID, "bimodalAfternoon", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    BimodalAfternoon = #T.

hasHormoneFeature(MasterID, "bimodalEvening", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    personFeaturesI(MasterID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening),
    BimodalEvening = #T.

hasHormoneFeature(MasterID, "biomodel", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    hasHormoneFeature(MasterID, "bimodalAfternoon", "7fca6ef2-8836-409b-a958-5287134aee49").

hasHormoneFeature(MasterID, "biomodel", "7fca6ef2-8836-409b-a958-5287134aee49") :-
    hasHormoneFeature(MasterID, "bimodalEvening", "7fca6ef2-8836-409b-a958-5287134aee49").

% personFeatures(ID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening
% ) :- personFeaturesI(ID,M,Am,Ac,P,N,PT,NT,Ph,Morn,After,Eve,Ni,MornPeak,AfterPeak,EvePeak,NiPeak,BimodalAfternoon,BimodalEvening
% ).

@bind("phenoTypeExport","csv useHeaders=true","./disk/assignment/data/csvs","phenotype.csv").
@bind("riskFactorOf","csv useHeaders=true","./disk/assignment/data/csvs","risk-factor.csv").
@bind("healthyPhenotypeOf","csv useHeaders=true","./disk/assignment/data/csvs","healthy-phenotype.csv").
@bind("hasHormoneFeature","csv useHeaders=true","./disk/assignment/data/csvs","has-hormone-feature.csv").
@bind("hasHormoneFeature","csv useHeaders=true","./disk/assignment/data/csvs","has-hormone-feature.csv").

@output("phenoTypeExport").
@output("riskFactorOf").
@output("healthyPhenotypeOf").
@output("hasHormoneFeature").

@output("healthyPhenotypesOf").
@output("riskFactorsOf").
@output("phenotypesOf").
@post("phenotypesOf", "orderby(1)").


[MESSAGE]> Can't connect to Vadalog server. [ERROR]> noVadalogServer

In [34]:
@bind("person","csv useHeaders=true","./disk/assignment/output/csvs","person.csv").
@bind("personFeaturesI","csv useHeaders=true","./disk/assignment/data/high-resolution","person_features.csv").

@model("person", "['PersonId(ID):int','gender:string','smoker:boolean','height:double','waist:double','hip:double','waist_hip_ratio:double','sysBP:int','diaBP:int','weight:double','BMI:double','age:int']").
@model("personFeatures", "['MasterID(ID):int','Mesor_hat:double','AmplitudeHat:double','AcrophaseHat:double','Peak:double','Nadir:double','PeakTime:double','NadirTime:double','Phase:double','Morning:double','Afternoon:double','Evening:double','Night:double','MorningPeak:double','AfternoonPeak:double','EveningPeak:double','NightPeak:double','BimodalAfternoon:boolean','BimodalEvening:boolean']").
@model("phenotype","['phenotype_id(ID):string']").

phenoType("BMIOverweight").
phenoType("BMIObese").
phenoType("BMIUnderweight").
phenoType("BMINormal").
phenoType("Smoker"). %
phenoType("NonSmoker"). %
phenoType("LowBloodPressure").
phenoType("HyperState2BP").
phenoType("HyperStage1BP").
phenoType("PreHyperBP").
phenoType("NormalBP").
phenoType("LowBP").
phenoType("LowWaistToHipRatio").
phenoType("MediumWaistToHipRatio").
phenoType("HighWaistToHipRatio").

phenoTypeExport(PhenoTypeId):- phenoType(PhenoTypeId).

@model("riskFactorOf","['RiskFactorId:phenotype','PersonId:person']").
riskFactorOf("BMIUnderweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B < 18.5.
@model("healthyPhenotypeOf","['PhenoTypeId:phenotype','PersonId:person']").
healthyPhenotypeOf("BMIHealthy",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 18.5, B < 25.
riskFactorOf("BMIOverweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 25.0, B < 30 .
riskFactorOf("BMIObese",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 30.0.
riskFactorOf("Smoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #T.
healthyPhenotypeOf("NonSmoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #F.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 160.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 140, Sys < 160, Dia < 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 90, Sys < 160, Dia < 100.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 120, Sys < 140, Dia < 90.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 80, Sys < 140, Dia < 90.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 90, Sys < 120, Dia < 80.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 60, Sys < 120, Dia < 80.
riskFactorOf("LowBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys < 90, Dia < 60.
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.8, G = "Female".
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.95, G = "Male".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.8, WH <= 0.85, G = "Female".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.95, WH < 1.0, G = "Male".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.85, G = "Female".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH >= 1.0, G = "Male".

healthyPhenotypesOf(MasterId,Phenotypes) :- healthyPhenotypeOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype).
riskFactorsOf(MasterId,Phenotypes,Count) :- riskFactorOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype), Count=mcount(Phenotype).
phenotypesOf(MasterId,Healthy,RiskFactors,RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).
phenotypesOf(MasterId,Healthy,{},RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), not riskFactorsOf(MasterId,RiskFactors),RiskCount = 0.
phenotypesOf(MasterId,{},RiskFactors,RiskCount) :- not healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).

@model("phenotypeCounts","['Phenotype:phenotype','Count:int']").
phenotypeCounts(Phenotype,Count) :- riskFactorOf(Phenotype,I),Count=mcount(I).
phenotypeCounts(Phenotype,Count) :- healthyPhenotypeOf(Phenotype,I),Count=mcount(I).
@output("phenotypeCounts").
@post("phenotypeCounts","orderby(2)").

Phenotype,Count
HyperStage2BP,2
BMIUnderweight,7
Smoker,26
HyperStage1BP,28
HighWaistToHipRatio,49
BMIOverweight,63
MediumWaistToHipRatio,63
PreHyperBP,68
LowWaistToHipRatio,102
NormalBP,115


In [65]:
@bind("person","csv useHeaders=true","./disk/assignment/output/csvs","person.csv").
@bind("personFeaturesI","csv useHeaders=true","./disk/assignment/data/high-resolution","person_features.csv").

@model("person", "['PersonId(ID):int','gender:string','smoker:boolean','height:double','waist:double','hip:double','waist_hip_ratio:double','sysBP:int','diaBP:int','weight:double','BMI:double','age:int']").
% @model("personFeatures", "['MasterID(ID):int','Mesor_hat:double','AmplitudeHat:double','AcrophaseHat:double','Peak:double','Nadir:double','PeakTime:double','NadirTime:double','Phase:double','Morning:double','Afternoon:double','Evening:double','Night:double','MorningPeak:double','AfternoonPeak:double','EveningPeak:double','NightPeak:double','BimodalAfternoon:boolean','BimodalEvening:boolean']").
@model("phenotype","['phenotype_id(ID):string']").

phenoType("BMIOverweight").
phenoType("BMIObese").
phenoType("BMIUnderweight").
phenoType("BMINormal").
phenoType("Smoker"). %
phenoType("NonSmoker"). %
phenoType("LowBloodPressure").
phenoType("HyperState2BP").
phenoType("HyperStage1BP").
phenoType("PreHyperBP").
phenoType("NormalBP").
phenoType("LowBP").
phenoType("LowWaistToHipRatio").
phenoType("MediumWaistToHipRatio").
phenoType("HighWaistToHipRatio").

phenoTypeExport(PhenoTypeId):- phenoType(PhenoTypeId).

@model("riskFactorOf","['RiskFactorId:phenotype','PersonId:person']").
riskFactorOf("BMIUnderweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B < 18.5.
@model("healthyPhenotypeOf","['PhenoTypeId:phenotype','PersonId:person']").
healthyPhenotypeOf("BMIHealthy",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 18.5, B < 25.
riskFactorOf("BMIOverweight",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 25.0, B < 30 .
riskFactorOf("BMIObese",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), B >= 30.0.
riskFactorOf("Smoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #T.
healthyPhenotypeOf("NonSmoker",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), S = #F.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 160.
riskFactorOf("HyperStage2BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 140, Sys < 160, Dia < 100.
riskFactorOf("HyperStage1BP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 90, Sys < 160, Dia < 100.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 120, Sys < 140, Dia < 90.
riskFactorOf("PreHyperBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 80, Sys < 140, Dia < 90.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys >= 90, Sys < 120, Dia < 80.
healthyPhenotypeOf("NormalBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Dia >= 60, Sys < 120, Dia < 80.
riskFactorOf("LowBP",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), Sys < 90, Dia < 60.
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.8, G = "Female".
healthyPhenotypeOf("LowWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH <= 0.95, G = "Male".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.8, WH <= 0.85, G = "Female".
riskFactorOf("MediumWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.95, WH < 1.0, G = "Male".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH > 0.85, G = "Female".
riskFactorOf("HighWaistToHipRatio",I) :- person(I,G,S,H,W,Hi,WH,Sys,Dia,Kg,B,A), WH >= 1.0, G = "Male".

healthyPhenotypesOf(MasterId,Phenotypes) :- healthyPhenotypeOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype).
riskFactorsOf(MasterId,Phenotypes,Count) :- riskFactorOf(Phenotype,MasterId), Phenotypes = munion({}|Phenotype), Count=mcount(Phenotype).
phenotypesOf(MasterId,Healthy,RiskFactors,RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).
phenotypesOf(MasterId,Healthy,{},RiskCount) :- healthyPhenotypesOf(MasterId,Healthy), not riskFactorsOf(MasterId,RiskFactors),RiskCount = 0.
phenotypesOf(MasterId,{},RiskFactors,RiskCount) :- not healthyPhenotypesOf(MasterId,Healthy), riskFactorsOf(MasterId,RiskFactors,RiskCount).

anyPhenotypeOf(Phenotype,MasterId) :- riskFactorOf(Phenotype,MasterId).
anyPhenotypeOf(Phenotype,MasterId) :- healthyPhenotypeOf(Phenotype,MasterId).
phenoTypeSum(Phenotype,Sum,Count) :- 
    anyPhenotypeOf(Phenotype,MasterId),
    personFeaturesI(MasterId,Mesor),
    Sum=msum(Mesor),
    Count=mcount(MasterId).
phenoTypeAverage(Phenotype,AverageMesor) :- phenoTypeSum(Phenotype,Sum,Count),AverageMesor=Sum/Count.

personFeatures(MasterID,AverageMesor) :- personFeaturesI(MasterID,Mesor).
% MasterID,Mesor_hat,AmplitudeHat,Peak,Nadir,PeakTime,NadirTime,Phase,Morning,Afternoon,Evening,Night,MorningPeak,AfternoonPeak,EveningPeak,NightPeak,BimodalAfternoon,BimodalEvening

    
@model("phenotypeCounts","['Phenotype:phenotype','Count:int']").
phenotypeCounts(Phenotype,Count) :- riskFactorOf(Phenotype,I),Count=mcount(I).
phenotypeCounts(Phenotype,Count) :- healthyPhenotypeOf(Phenotype,I),Count=mcount(I).
% @output("phenotypeCounts").
@post("phenoTypeAverage","orderby(-2)").
@output("phenoTypeAverage").
% @output("personFeatures").

Phenotype,AverageMesor
HyperStage2BP,5.514
BMIUnderweight,4.87186
BMIHealthy,4.1215
NormalBP,4.03339
NonSmoker,3.95449
MediumWaistToHipRatio,3.94178
LowWaistToHipRatio,3.92794
PreHyperBP,3.83131
HighWaistToHipRatio,3.82094
HyperStage1BP,3.59271
